# Volcano Image Scraping in Different Languages

**Note**: This notebook is relevant for testing model training with a custom class (volcano-related images).

This notebook collects the first 20 images returned by Google Images for the word "volcano" in four languages:

- Spanish: `volcán`
- English: `volcano`
- German: `Vulkan`
- Russian: `вулкан`

Images are saved locally in folders organized by language within `volcanes_scrapping/images`. 

---
### Steps
1. Set up environment (install dependencies if missing)
2. Initialize WebDriver (Chrome Headless or Firefox)
3. Execute search for each language
4. Scroll and collect image URLs
5. Download the first 20 images per language
6. Visualize a sample

---
### Requirements
Make sure you have previously installed:
```n
pip install selenium webdriver-manager requests Pillow
```n
(Already added to `requirements.txt`).

In [ ]:
# (Optional) Quick dependency check
# If any are missing, uncomment and install.
# !pip install selenium webdriver-manager requests Pillow

import os
import time
import math
import random
from pathlib import Path
from typing import List, Set

import requests
from PIL import Image
from io import BytesIO

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.chrome import ChromeDriverManager
from webdriver_manager.firefox import GeckoDriverManager

BASE_OUTPUT_DIR = Path('images')
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANG_QUERIES = {
    'es': 'volcán lava',
    'en': 'volcano lava',
    'de': 'Vulkan Lava',
    'ru': 'вулкан лава'
}

MAX_IMAGES_PER_LANG = 20
SCROLL_PAUSE = 1.0  # seconds
MAX_SCROLLS = 12     # limit to avoid infinite loops
USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
    '(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36'
)

def setup_driver(headless: bool = True):
    """Create a WebDriver using Chrome headless; if it fails, fall back to Firefox."""
    # Attempt with Chrome
    try:
        chrome_options = ChromeOptions()
        if headless:
            chrome_options.add_argument('--headless=new')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--log-level=3')
        chrome_options.add_argument(f'user-agent={USER_AGENT}')
        chrome_options.add_argument('--window-size=1920,1080')
        service = ChromeService(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        return driver
    except Exception as e:
        print(f"Chrome failed: {e}. Trying Firefox...")
        # Firefox fallback
        ff_options = FirefoxOptions()
        if headless:
            ff_options.add_argument('--headless')
        ff_options.set_preference('general.useragent.override', USER_AGENT)
        ff_service = FirefoxService(GeckoDriverManager().install())
        driver = webdriver.Firefox(service=ff_service, options=ff_options)
        return driver

def google_image_search_url(query: str) -> str:
    from urllib.parse import quote_plus
    return f"https://www.google.com/search?tbm=isch&q={quote_plus(query)}"


def fetch_image_elements(driver) -> List[Image]:
    # Returns potential img elements
    return driver.find_elements(By.CSS_SELECTOR, 'img')


def extract_image_urls(driver, max_links: int) -> Set[str]:
    """Scroll and collect src/data-src of images up to max_links."""
    image_urls: Set[str] = set()
    scrolls = 0
    last_height = driver.execute_script('return document.body.scrollHeight')

    while len(image_urls) < max_links and scrolls < MAX_SCROLLS:
        # Get current elements
        imgs = fetch_image_elements(driver)
        for img in imgs:
            src = img.get_attribute('src') or img.get_attribute('data-src')
            if not src:
                continue
            if src.startswith('data:'):
                # ignore inline base64
                continue
            if src.startswith('http'):
                image_urls.add(src)
                if len(image_urls) >= max_links:
                    break
        # Scroll down
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(SCROLL_PAUSE + random.uniform(0, 0.5))
        new_height = driver.execute_script('return document.body.scrollHeight')
        if new_height == last_height:
            # Attempt to load more results by clicking the button if it exists
            try:
                more_button = driver.find_element(By.CSS_SELECTOR, 'input[type="button"], span[role="button"]')
                more_button.click()
                time.sleep(2)
            except Exception:
                # No more results
                break
        last_height = new_height
        scrolls += 1
    return image_urls


def download_image(url: str, out_path: Path, timeout: int = 10) -> bool:
    try:
        headers = {'User-Agent': USER_AGENT}
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code != 200:
            return False
        img_bytes = resp.content
        # Quick validation by attempting to open with PIL
        img = Image.open(BytesIO(img_bytes))
        img_format = img.format or 'JPEG'
        out_file = out_path.with_suffix('.' + img_format.lower())
        img.save(out_file)
        return True
    except Exception:
        return False


def download_images(urls: Set[str], lang_code: str):
    lang_dir = BASE_OUTPUT_DIR / lang_code
    lang_dir.mkdir(parents=True, exist_ok=True)
    success = 0
    for i, url in enumerate(sorted(urls)):
        out_path = lang_dir / f"{lang_code}_{i+1}"
        if download_image(url, out_path):
            success += 1
        else:
            print(f"   Download failed: {url[:60]}...")
    print(f"Downloaded {success}/{len(urls)} images for {lang_code} in {lang_dir}")

print("Driver starting...")
DRIVER = setup_driver(headless=True)
print("Driver ready.")

Driver starting...
Driver ready.
Driver ready.


In [ ]:

TOPICS = {
    'lava': {
        'es': 'volcán lava',
        'en': 'volcano lava',
        'de': 'Vulkan Lava',
        'ru': 'вулкан лава'
    },
    'ceniza': {
        'es': 'volcán ceniza',  # could also use 'ceniza volcánica' (volcanic ash)
        'en': 'volcano ash',
        'de': 'Vulkan Asche',   # 'Vulkanasche' is the compound word
        'ru': 'вулкан пепел'
    },
    'erupcion': {
        'es': 'volcán erupción',  # also 'erupción volcánica' (volcanic eruption)
        'en': 'volcano eruption',
        'de': 'Vulkan Eruption',
        'ru': 'вулкан извержение'
    }
}

SELECTED_TOPICS = ['lava', 'ceniza', 'erupcion']

from pathlib import Path

def download_images_topic(urls, lang_code: str, topic_slug: str):
    topic_dir = BASE_OUTPUT_DIR / topic_slug / lang_code
    topic_dir.mkdir(parents=True, exist_ok=True)
    success = 0
    for i, url in enumerate(sorted(urls)):
        out_path = topic_dir / f"{topic_slug}_{lang_code}_{i+1}"
        if download_image(url, out_path):
            success += 1
        else:
            print(f"   Download failed: {url[:70]}...")
    print(f"Downloaded {success}/{len(urls)} images for {lang_code} in {topic_dir}")

for topic in SELECTED_TOPICS:
    print(f"\n############################")
    print(f"### Topic: {topic}")
    print(f"############################")
    # Create new driver per topic (avoids long sessions)
    driver_local = setup_driver(headless=True)
    try:
        for lang, query in TOPICS[topic].items():
            print(f"\n=== Searching images for '{query}' ({lang}) [{topic}] ===")
            url = google_image_search_url(query)
            driver_local.get(url)
            time.sleep(2)
            urls = extract_image_urls(driver_local, MAX_IMAGES_PER_LANG)
            print(f"Obtained {len(urls)} URLs")
            download_images_topic(urls, lang, topic)
    finally:
        driver_local.quit()
        print(f"Driver closed for topic {topic}.")

print("\nAdditional scraping completed.")


############################
### Topic: lava
############################

=== Searching images for 'volcán lava' (es) [lava] ===

=== Searching images for 'volcán lava' (es) [lava] ===
Obtained 20 URLs
Obtained 20 URLs
   Download failed: https://fonts.gstatic.com/s/i/productlogos/googleg/v6/24px.svg...
Downloaded 19/20 images for es in volcanes_scrapping\images\lava\es

=== Searching images for 'volcano lava' (en) [lava] ===
   Download failed: https://fonts.gstatic.com/s/i/productlogos/googleg/v6/24px.svg...
Downloaded 19/20 images for es in volcanes_scrapping\images\lava\es

=== Searching images for 'volcano lava' (en) [lava] ===
Obtained 20 URLs
Obtained 20 URLs
   Download failed: https://fonts.gstatic.com/s/i/productlogos/googleg/v6/24px.svg...
Downloaded 19/20 images for en in volcanes_scrapping\images\lava\en

=== Searching images for 'Vulkan Lava' (de) [lava] ===
   Download failed: https://fonts.gstatic.com/s/i/productlogos/googleg/v6/24px.svg...
Downloaded 19/20 images for

In [ ]:
# Merge all images into a single folder encompassing all topics and languages
final_dir = BASE_OUTPUT_DIR / 'all_images'
final_dir.mkdir(parents=True, exist_ok=True)
for topic in SELECTED_TOPICS:
    topic_path = BASE_OUTPUT_DIR / topic
    for lang_dir in topic_path.iterdir():
        if lang_dir.is_dir():
            for img_file in lang_dir.iterdir():
                if img_file.is_file():
                    dest_file = final_dir / img_file.name
                    # Avoid overwriting
                    count = 1
                    while dest_file.exists():
                        dest_file = final_dir / f"{img_file.stem}_{count}{img_file.suffix}"
                        count += 1
                    img_file.rename(dest_file)
print(f"\nAll images merged in {final_dir}")


All images merged in volcanes_scrapping\images\all_images


## Manual Image Cleanup

After downloading all images, a manual review and cleanup process is performed to remove irrelevant or low-quality images. This ensures that only appropriate volcano-related images are retained for model training purposes.